# Test TcLab PID real time


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from matplotlib import colors as mcolors

import package_DBR
from package_DBR import myRound, SelectPath_RT, Delay_RT, FO_RT, FOPDT, SOPDT, FOPDT_cost, SOPDT_cost, Process, Bode
import package_LAB
from package_LAB import LL_RT, PID_RT, IMC_tuning


Ts = 1.0
Tsim = 3000
t = np.arange(0, Tsim, Ts)

PV_0 = 42
DV_0 = 50
MV_0 = 50

SPPath = {0: PV_0, 1000: PV_0 - 5}
ManPath = {0: True, 500: False}
MVManPath = {0:PV_0+10}
DVPatch = {0: DV_0, 2000: DV_0+10}
FF = True
ManFF = False

# Modèles (issus de tes identifications)
K_p, T1_p, T2_p, theta_p = 0.32, 149.5, 15.0, 4.0   # Procédé (H1 -> T1)
K_d, T1_d, T2_d, theta_d = 0.36, 170.77, 6.12, 5.43  # Perturbation (H2 -> T1)

# Réglages PID
# DISCUTE ALPHA ET GAMMA
alpha = 1
Kc, Ti, Td = IMC_tuning(K_p, T1_p, T2_p, theta_p, 0.7)
print("Kc",Kc, "Ti",Ti, "Td",Td)
MVMin, MVMax = 0, 100

t = []

E = []
EP = []
ED = []
PV = []
SP = []
DV = []
MV = []
MVP =[]
MVI =[]
MVD =[]
MVFF_Delay = []
MVFF_LL = []
MVFF = []
MVMan = []
Man = []


MV_Delayp = []
PV_FOp = []
PV_p = []

MV_Delayd = []
PV_FOd = []
PV_d = []

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios':[1, 4, 4, 1]})

# --- Graph 1 : État Manuel vs Auto (Man) ---
ax1.step([0,1],[0,100], color='black', label='Man', where='post', linewidth=1.5)
ax1.set_ylabel('Value of Man [0 or 1]')
ax1.set_ylim([-0.2, 1.2])
ax1.set_yticks([0, 1])
ax1.legend(loc='upper right', frameon=False)
ax1.set_title(f"Closed-loop response with PID controller and feedforward", fontsize=11)

ax2.axhline(0, color='salmon', linestyle='--', linewidth=1.5)
ax2.set_ylabel('Value of MV [%]')
ax2.legend(loc='upper right', frameon=False)

l2_MV,   = ax2.plot([], [], color='blue', label='MV', linewidth=2)
l2_MVP,  = ax2.plot([], [], color='violet', linestyle=':', label='MVP', alpha=0.8)
l2_MVI,  = ax2.plot([], [], color='mediumturquoise', linestyle=':', label='MVI')
l2_MVD,  = ax2.plot([], [], color='navy', linestyle='--', label='MVD', alpha=0.7)
l2_MVFF, = ax2.plot([], [], color='salmon', linestyle='--', label='MVFF')

for i in range(Tsim):
    t.append(i)
    SelectPath_RT(SPPath, t, SP)
    SelectPath_RT(DVPatch, t, DV)

    Delay_RT(DV - DV_0*np.ones_like(DV), theta_d, Ts, MVFF_Delay, 0)

    #def LL_RT(MV,Kp,Tlag,Tlead,Ts,PV,PVInit=0,method='EBD'):
    LL_RT(MVFF_Delay, -K_d/K_p, T1_p, T1_d, Ts, MVFF_LL, 0)
    if FF:
        LL_RT(MVFF_LL, 1.0, T2_p, T2_d, Ts, MVFF, 0)
    else:
        MVFF.append(0)

    SelectPath_RT(MVManPath, t, MVMan)
    SelectPath_RT(ManPath, t, Man)

    PID_RT(SP, PV, Man, MVMan, MVFF, Kc, Ti, Td, alpha, Ts, MVMin, MVMax, MV, MVP, MVI, MVD, E, ManFF, PV_0)

    #input-output dynamics
    Delay_RT(MV, theta_p, Ts, MV_Delayp, MV_0)
    FO_RT(MV_Delayp, K_p, T1_p, Ts, PV_FOp)
    FO_RT(PV_FOp, 1, T2_p, Ts, PV_p)

    Delay_RT(DV - DV_0*np.ones_like(DV), theta_d, Ts, MV_Delayd)
    FO_RT(MV_Delayd, K_d, T1_d, Ts, PV_FOd)
    FO_RT(PV_FOd, 1, T2_d, Ts, PV_d)

    PV_ = PV_p[-1]+ PV_d[-1]

    PV.append(PV_ +PV_0-K_p*MV_0)



# Configuration des proportions (Man petit, MV grand, PV grand, DV petit)
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios':[1, 4, 4, 1]})

# Couleur de fond gris clair pour la figure et les axes
bg_color = '#E8E8E8'
fig.patch.set_facecolor(bg_color)

for ax in (ax1, ax2, ax3, ax4):
    ax.set_facecolor(bg_color)
    # ax.grid(True, alpha=0.3) # Décommentez si vous voulez une grille légère

# --- Graph 1 : État Manuel vs Auto (Man) ---
ax1.step(t, np.array(Man).astype(int), color='black', label='Man', where='post', linewidth=1.5)
ax1.set_ylabel('Value of Man [0 or 1]')
ax1.set_ylim([-0.2, 1.2])
ax1.set_yticks([0, 1])
ax1.legend(loc='upper right', frameon=False)
ax1.set_title(f"Closed-loop response with PID controller and feedforward", fontsize=11)

# --- Graph 2 : Signaux de commande (MV et ses composantes) ---
# Ligne zéro en pointillé rouge
ax2.axhline(0, color='salmon', linestyle='--', linewidth=1.5)

ax2.plot(t, MV, color='blue', label='MV', linewidth=2)
ax2.plot(t, MVP, color='violet', linestyle=':', label='MVP', alpha=0.8)
ax2.plot(t, MVI, color='mediumturquoise', linestyle=':', label='MVI')
ax2.plot(t, MVD, color='navy', linestyle='--', label='MVD', alpha=0.7)
ax2.plot(t, MVFF, color='salmon', linestyle='--', label='MVFF')



# --- Graph 3 : Températures (SP, PV) ---
ax3.step(t, SP, color='firebrick', label='SP', where='post', linewidth=2)
ax3.plot(t, PV, color='darkgreen', label='PV', linewidth=2)

ax3.set_ylabel('Value of PV [°C]')
ax3.legend(loc='upper right', frameon=False)

# --- Graph 4 : Perturbation (DV) ---
ax4.step(t, DV, color='red', label='DV', where='post', linewidth=1.5)
ax4.set_ylabel('Value of DV [%]')
ax4.set_xlabel('Time [s]')
ax4.set_xlim([0, Tsim])

# La légende du DV sur l'image semble être en haut à gauche
ax4.legend(loc='upper left', frameon=False)

plt.tight_layout()
plt.subplots_adjust(hspace=0.15) # Réduit un peu l'espace entre les graphiques
plt.show()